# DragonRadio in Colab: feasibility probe

This notebook answers one question before any teaching notebooks are written: **can the real DragonRadio
stack run in simulation on a Colab runtime and keep up with real time?**

The simulator runs the production PHY, synthesizer, channelizer, ARQ, queues and TDMA MAC. Only the radio device
is simulated. Runs are paced by the wall clock, so if the runtime's CPUs fall behind, transmit bursts reach the
simulated air late, receive buffers overflow, and the run fails. Those failures are counted.

The notebook:

1. records the runtime (OS, glibc, CPUs);
2. installs a self-contained DragonRadio bundle (its own Python 3.12; nothing from Colab's Python is used);
3. runs the `clean-arq` and `saturation` scenarios several times while sampling CPU load;
4. reports late and dropped bursts and the **lead time**: how long before its air time each burst reached the
   simulated air. Lead near zero means the runtime is barely keeping up.

**Decision rule.** Any late, underflowed or dropped burst fails a run. If every run passes, Colab can host live
runs. The lead times show the margin. For reference, on a 2.6 GHz Xeon pinned to one core's two hyperthreads
(gridgpu01, 2026-11-08) all six runs passed, the smallest lead was 1.0–2.2 ms, and the median was about 8 ms.
The same scenario also passed on a single logical CPU.

Runtime → *Change runtime type* → CPU is enough. Record which tier you are on (free or Pro).

## 1. The runtime

In [ ]:
import os, platform, subprocess, sys, json, time, threading, shutil
from pathlib import Path

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

runtime = {
    'os': sh(". /etc/os-release; echo $PRETTY_NAME"),
    'glibc': sh("ldd --version | head -1 | awk '{print $NF}'"),
    'kernel': platform.release(),
    'cpus_usable': len(os.sched_getaffinity(0)),
    'cpus_host': os.cpu_count(),
    'cpu_model': sh("grep -m1 'model name' /proc/cpuinfo | cut -d: -f2").strip(),
    'threads_per_core': sh("lscpu | awk -F: '/Thread\\(s\\) per core/{print $2}'").strip(),
    'mem_gb': round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1),
    'notebook_python': sys.version.split()[0],
}
for k, v in runtime.items():
    print(f'{k:>17}: {v}')

## 2. Install the bundle

The bundle (`drsim.tar.gz`) is built by `grid/build-colab-bundle.sh` on Ubuntu 22.04. Give it to the notebook in
one of three ways: a download URL, a file in your Google Drive, or an upload from this computer.

In [ ]:
BUNDLE_URL = ''                                         # e.g. a GitHub release asset
DRIVE_PATH = '/content/drive/MyDrive/drsim.tar.gz'      # used if BUNDLE_URL is empty
INSTALL_DIR = Path('/content')

tarball = INSTALL_DIR / 'drsim.tar.gz'
if (INSTALL_DIR / 'drsim' / 'env.sh').exists():
    print('bundle already installed')
else:
    if BUNDLE_URL:
        subprocess.run(['curl', '-sfL', BUNDLE_URL, '-o', str(tarball)], check=True)
    else:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as e:
            print('Drive not mounted:', e)
        if Path(DRIVE_PATH).exists():
            shutil.copy(DRIVE_PATH, tarball)
        else:
            from google.colab import files
            print('Choose drsim.tar.gz to upload')
            up = files.upload()
            shutil.move(next(iter(up)), tarball)
    subprocess.run(['tar', '-xzf', str(tarball), '-C', str(INSTALL_DIR)], check=True)

DRSIM = INSTALL_DIR / 'drsim'
print(sh(f'head -4 {DRSIM}/MANIFEST.txt'))

In [ ]:
# Environment for the bundle's processes only; the notebook kernel is untouched.
ENV = dict(os.environ)
ENV.update({
    'PYTHONHOME': str(DRSIM / 'python'),
    'PYTHONPATH': str(DRSIM / 'dragonradio' / 'python' / 'dragonradio'),
    'LD_LIBRARY_PATH': f"{DRSIM}/lib:{DRSIM}/python/lib",
    'PATH': f"{DRSIM}/bin:{DRSIM}/python/bin:" + os.environ['PATH'],
})
PY = str(DRSIM / 'python' / 'bin' / 'python3')
RADIO = str(DRSIM / 'bin' / 'dragonradio')

# Smoke test: every shared library resolves, and the embedded interpreter can import the radio module.
missing = sh(f'LD_LIBRARY_PATH={ENV["LD_LIBRARY_PATH"]} ldd {RADIO} | grep "not found"')
print('missing libraries:', missing or 'none')
probe = Path('/content/smoke.py')
probe.write_text('import dragonradio.radio as r, numpy, sys\nprint("embedded Python", sys.version.split()[0], "radio module OK")\n')
r = subprocess.run([RADIO, str(probe)], env=ENV, capture_output=True, text=True, timeout=120)
print(r.stdout[-2000:], r.stderr[-2000:])

## 3. Run scenarios and sample CPU load

`run_scenario` starts the existing harness (`scripts/sim-fullstack.py`): the air relay plus one `dragonradio`
process per node. Each run is about 30 s of wall-clock time (4 s warm-up, 20 s traffic, 4 s drain, plus start-up).

In [ ]:
import psutil
CPUS = sorted(os.sched_getaffinity(0))   # on Colab: all of them

RESULTS = Path('/content/probe-results')

def radio_wrapper(nice):
    # Lower the radios' priority so the air relay (one Python process) is not starved.
    w = Path(f'/content/dragonradio-nice{nice}.sh')
    w.write_text(f'#!/bin/sh\nexec nice -n {nice} {RADIO} "$@"\n'); w.chmod(0o755)
    return str(w)

def run_scenario(scenario, tag, radio_args=(), nice=None):
    out = RESULTS / f'{scenario}-{tag}'
    shutil.rmtree(out, ignore_errors=True)
    cpu, stop = [], threading.Event()
    def sample():
        t0 = time.monotonic()
        while not stop.is_set():
            before = psutil.cpu_times()
            per = psutil.cpu_percent(interval=0.5, percpu=True)
            after = psutil.cpu_times()
            total = sum(after) - sum(before)
            steal = 100 * (getattr(after, 'steal', 0) - getattr(before, 'steal', 0)) / total if total > 0 else 0.0
            cpu.append((time.monotonic() - t0, [per[i] for i in CPUS], steal))
    th = threading.Thread(target=sample, daemon=True); th.start()
    t0 = time.monotonic()
    p = subprocess.run([PY, str(DRSIM / 'dragonradio' / 'scripts' / 'sim-fullstack.py'),
                        '--dragonradio', RADIO if nice is None else radio_wrapper(nice),
                        '--outdir', str(out), '--scenario', scenario, '--', *radio_args],
                       env=ENV, capture_output=True, text=True, cwd=str(DRSIM / 'dragonradio'))
    wall = time.monotonic() - t0
    stop.set(); th.join()
    summary_path = out / scenario / 'summary.json'
    if not summary_path.exists():
        summary_path = next(out.rglob('summary.json'), None)
    s = json.loads(summary_path.read_text()) if summary_path else {'passed': False, 'error': p.stdout[-3000:] + p.stderr[-3000:]}
    s['_wall_s'], s['_cpu'], s['_dir'] = wall, cpu, str(summary_path.parent if summary_path else out)
    return s

def brief(s):
    dev = s.get('device', {})
    leads = []
    bl = Path(s['_dir']) / 'bursts.csv'
    if bl.exists():
        import pandas as pd
        leads = pd.read_csv(bl)['lead'].to_numpy() * 1e3
    import numpy as np
    return {
        'passed': s.get('passed'),
        'tx_late': sum(d.get('tx_late', 0) for d in dev.values()),
        'rx_late': sum(d.get('rx_late', 0) for d in dev.values()),
        'rx_partial_late': sum(d.get('rx_partial_late', 0) for d in dev.values()),
        'rx_overflows': sum(d.get('rx_overflows', 0) for d in dev.values()),
        'rx_queue_drops': sum(d.get('rx_queue_drops', 0) for d in dev.values()),
        'delivery_min': min((f['delivery'] for f in s.get('flows', []) if f.get('delivery') is not None), default=None),
        'late_bursts': int(np.sum(leads < 0)) if len(leads) else None,
        'bursts': len(leads),
        'frame_delivery_min': min((l['frame_delivery'] for l in s.get('links_accounting', {}).values()
                                   if l.get('frame_delivery') is not None), default=None),
        'failed_checks': [c['check'] for c in s.get('checks', []) if not c['passed']] or s.get('error', ''),
        'lead_min_ms': float(np.min(leads)) if len(leads) else None,
        'lead_p1_ms': float(np.percentile(leads, 1)) if len(leads) else None,
        'lead_median_ms': float(np.median(leads)) if len(leads) else None,
        'cpu_mean_%': float(np.mean([np.mean(c[1]) for c in s['_cpu']])) if s['_cpu'] else None,
        'steal_mean_%': float(np.mean([c[2] for c in s['_cpu']])) if s['_cpu'] else None,
        'steal_max_%': float(np.max([c[2] for c in s['_cpu']])) if s['_cpu'] else None,
        'wall_s': round(s['_wall_s'], 1),
    }

first = run_scenario('clean-arq', 'baseline')
print(json.dumps(brief(first), indent=1))
if not first.get('passed'):
    for c in first.get('checks', []):
        print('PASS' if c['passed'] else 'FAIL', c['check'])
    print(first.get('error', ''))

## 4. Timing margin

Left: lead time of every burst (how early it reached the simulated air). Bursts with lead below zero were late.
The three peaks are the three frames of each TDMA slot, all handed to the device when the slot is prepared.
Right: CPU load per logical CPU during the run, and steal time (dashed): the share of time this VM was ready to
run but its host ran something else. Expect it near 100%: some radio threads poll while they wait for
work, so this measures polling as well as processing. Late bursts and lead are the measure that counts.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def plot_margin(s, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
    bl = Path(s['_dir']) / 'bursts.csv'
    if bl.exists():
        b = pd.read_csv(bl)
        for node, g in b.groupby('node'):
            ax[0].hist(g['lead'] * 1e3, bins=60, alpha=0.6, label=f'node {node}')
        ax[0].axvline(0, color='k', lw=1)
        ax[0].set_xlabel('lead before air time (ms)'); ax[0].set_ylabel('bursts'); ax[0].legend()
    t = [c[0] for c in s['_cpu']]
    per = np.array([c[1] for c in s['_cpu']])
    if len(per):
        for i in range(per.shape[1]):
            ax[1].plot(t, per[:, i], lw=1, label=f'cpu{CPUS[i]}')
        ax[1].plot(t, [c[2] for c in s['_cpu']], 'k--', lw=1, label='steal')
        ax[1].set_ylim(0, 105); ax[1].set_xlabel('s since start'); ax[1].set_ylabel('% busy')
        ax[1].legend(ncol=4, fontsize=8)
    fig.suptitle(title); fig.tight_layout(); plt.show()

plot_margin(first, 'clean-arq, run 1')

## 5. Compare settings

Two earlier Colab runs (2026-11-08: 2 vCPUs, Xeon @ 2.20 GHz) found two timing problems.

1. **The air relay** (one Python process) sometimes forwarded bursts 2–7 ms after their air time. Receivers
   release samples 2 ms after air time by default, so those bursts were partly lost. Waiting 10 ms
   (`--sim-rx-latency 0.010`) fixed this, and running the radios at `nice 10` cut relay lateness further.
2. **A radio** occasionally missed its 5 ms send lead (`tx_late`), and the device dropped that burst, which is one
   TDMA slot (3 frames): 1–2 slots out of about 900 per node per minute.

This run compares, three times on each scenario:

* **rx10+nice**: the best setting from run 2, for comparison in the same session.
* **rx10+nice+lead10**: the same, with the MAC handing each slot to the device 10 ms ahead instead of 5 ms
  (`--set slot_send_lead_time=0.010`). This adds about 5 ms to every latency.

Steal time is recorded per run, to tell host pauses apart from the radios' own load.

`REPEATS = 3` takes about 15 minutes.

In [ ]:
CONFIGS = {
    'rx10+nice':        dict(radio_args=['--sim-rx-latency', '0.010'], nice=10),
    'rx10+nice+lead10': dict(radio_args=['--sim-rx-latency', '0.010', '--set', 'slot_send_lead_time=0.010'], nice=10),
}
REPEATS = 3
rows = [{'config': 'baseline', 'scenario': 'clean-arq', 'run': 1, **brief(first)}]
print(rows[-1])
for name, cfg in CONFIGS.items():
    for scenario in ['clean-arq', 'saturation']:
        for i in range(REPEATS):
            s = run_scenario(scenario, f'{name}-r{i+1}', **cfg)
            rows.append({'config': name, 'scenario': scenario, 'run': i + 1, **brief(s)})
            print(rows[-1])
table = pd.DataFrame(rows)
table[['config', 'scenario', 'run', 'passed', 'late_bursts', 'bursts', 'frame_delivery_min', 'lead_min_ms',
       'lead_median_ms', 'rx_partial_late', 'tx_late', 'rx_late', 'steal_mean_%', 'steal_max_%', 'wall_s']]

In [ ]:
verdicts = {}
for name, g in table[table['config'] != 'baseline'].groupby('config'):
    verdicts[name] = ('LIVE RUNS OK' if g['passed'].all() else 'FAILS') + \
        f" ({int(g['passed'].sum())}/{len(g)} passed, smallest lead {g['lead_min_ms'].min():.2f} ms)"
verdict = '; '.join(f'{k}: {v}' for k, v in verdicts.items())
report = {'runtime': runtime, 'verdict': verdict, 'runs': rows}
Path('/content/probe-report.json').write_text(json.dumps(report, indent=1, default=str))
print(verdict)
print('Saved /content/probe-report.json: download it and send it back.')